In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

In [4]:
X,W,b,num_movies,num_features,num_users = load_precalc_params_small()
Y,R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [5]:
tsmean = np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Average rating for movie 1 : 3.400 / 5


In [33]:
def cofi_cost_func(X, W, b, Y, R, lambda_):
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J
    

In [34]:
num_users_r = 4
num_movies_r = 5
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = tf.reshape(b[0, :num_users_r], (1, -1))
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]



In [35]:
# Evaluate cost function
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# Evaluate cost function with regularization 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost: 5.86
Cost (with regularization): 30.30


In [42]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)

my_ratings[929] = 5   # Lord of the Rings
my_ratings[246] = 5   # Shrek
my_ratings[2716] = 3  # Inception
my_ratings[1150] = 5  # Incredibles
my_ratings[382] = 2   # Amelie
my_ratings[366] = 5   # Harry Potter 1
my_ratings[622] = 5   # Harry Potter 2
my_ratings[988] = 3   # Eternal Sunshine
my_ratings[2925] = 1  # Louis Theroux
my_ratings[2937] = 1  # Nothing to Declare
my_ratings[793] = 5   # Pirates of the Caribbean

In [43]:
Y, R = load_ratings_small()

Y = np.c_[my_ratings,Y]

R = np.c_[(my_ratings != 0).astype(int), R]

Ynorm, Ymean = normalizeRatings(Y, R)

In [44]:
num_movies, num_users = Y.shape
num_features = 100

tf.random.set_seed(1234)
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

optimizer =  keras.optimizers.Adam(1e-1)

In [48]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    with tf.GradientTape() as tape:
        cost_value = cofi_cost_func(X, W, b ,Ynorm, R, lambda_)

    grads = tape.gradient(cost_value, [X,W,b])

    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    if iter % 20 == 0:
        print(f"TRAINING LOSS AT ITERATION {iter}:{cost_value:0.1f}")

TRAINING LOSS AT ITERATION 0:2321157.1
TRAINING LOSS AT ITERATION 20:136168.5
TRAINING LOSS AT ITERATION 40:51862.5
TRAINING LOSS AT ITERATION 60:24597.5
TRAINING LOSS AT ITERATION 80:13629.5
TRAINING LOSS AT ITERATION 100:8487.1
TRAINING LOSS AT ITERATION 120:5807.3
TRAINING LOSS AT ITERATION 140:4311.4
TRAINING LOSS AT ITERATION 160:3435.1
TRAINING LOSS AT ITERATION 180:2901.9


In [49]:
#Recommendations

In [51]:
p = np.matmul(X.numpy(), np.transpose(W.numpy()))+ b.numpy()
pm = p + Ymean
my_predictions = pm[:,0]
ix = tf.argsort(my_predictions, direction='DESCENDING')
for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 4.58 for movie Memento (2000)
Predicting rating 4.43 for movie My Sassy Girl (Yeopgijeogin geunyeo) (2001)
Predicting rating 4.42 for movie Martin Lawrence Live: Runteldat (2002)
Predicting rating 4.42 for movie Delirium (2014)
Predicting rating 4.42 for movie Laggies (2014)
Predicting rating 4.42 for movie One I Love, The (2014)
Predicting rating 4.40 for movie Particle Fever (2013)
Predicting rating 4.39 for movie Eichmann (2007)
Predicting rating 4.39 for movie Battle Royale 2: Requiem (Batoru rowaiaru II: Chinkonka) (2003)
Predicting rating 4.39 for movie Into the Abyss (2011)
Predicting rating 4.37 for movie Son of the Bride (Hijo de la novia, El) (2001)


Original vs Predicted ratings:

Original 5.0, Predicted 4.88 for Shrek (2001)
Original 5.0, Predicted 4.84 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Original 2.0, Predicted 2.13 for Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Original 5.0, Predic

In [52]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

,pred,mean rating,number of ratings,title
2112,4.067851,4.238255,149,"Dark Knight, The (2008)"
155,3.978319,4.155914,93,Snatch (2000)
211,4.580973,4.122642,159,Memento (2000)
929,4.860788,4.118919,185,"Lord of the Rings: The Return of the King, The..."
653,4.104019,4.021277,188,"Lord of the Rings: The Two Towers, The (2002)"
2804,4.359065,3.989362,47,Harry Potter and the Deathly Hallows: Part 1 (...
773,4.260202,3.960993,141,Finding Nemo (2003)
1771,4.206928,3.944444,81,Casino Royale (2006)
2649,3.911139,3.943396,53,How to Train Your Dragon (2010)
2455,4.004590,3.887931,58,Harry Potter and the Half-Blood Prince (2009)
